# VibeCrop AI on Google Colab

Run the cells **in order, top to bottom**. What happens, and how long it takes:

| Cell | What it does | Time |
|---|---|---|
| 1 | Connect Google Drive (sounds and the noise model are kept there) | seconds |
| 2 | Install ffmpeg, fonts and the app | 1-2 min **every new session** |
| 3 | Read your Groq / Gemini keys from Colab **Secrets** | seconds |
| 4 | Download the sound-effect library **once** (stays on Drive) | a few min, first time only |
| 5 | Start the app and print a link | ~15 s |

**Before you start:** open the key icon in the left bar (Secrets), add `GROQ_API_KEY` and `GEMINI_API_KEY`, and switch
"Notebook access" on for both. Pick a GPU runtime (Runtime > Change runtime type > T4) if you want faster exports.

Good to know: the link changes every session; the session stops after a while idle (free Colab); anyone who has the
link can use the app **and your API keys' quota**, so do not share it. Colab is fine for trying things out and for
personal use; a permanent deployment needs a small server (the same code runs there).

In [ ]:
# 1. Google Drive: the sound library and the noise-removal model are saved here, so they survive between sessions
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/VibeCrop'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive folder:', DRIVE_DIR)

In [ ]:
# 2. Install (about 1-2 minutes). ffmpeg and the fonts come from apt; the app from GitHub.
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-core fonts-liberation > /dev/null
# Bold-pop captions use the Impact font; this is best effort (it is downloaded from a third-party mirror).
!(echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true | debconf-set-selections && apt-get -qq install -y ttf-mscorefonts-installer > /dev/null) || echo "Impact font not installed: bold-pop captions will use a default bold font."
!fc-cache -f > /dev/null
import os
REPO = '/content/vibecrop-ai'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/apoorvcurry21/vibecrop-ai.git {REPO}
else:
    !git -C {REPO} pull -q
!pip install -q -r {REPO}/requirements.txt
!ffmpeg -version | head -1

In [ ]:
# 3. Your API keys, from Colab Secrets (the key icon in the left bar). Only the NAMES are printed.
import sys, os
sys.path.insert(0, REPO + '/colab')
import colab_setup
keys, missing = colab_setup.load_secrets()
os.environ.update(keys)
print('Found:', sorted(keys), '| missing:', missing)
assert not missing, 'Add the missing secret(s) in the left bar (key icon), switch Notebook access on, then run this cell again.'
ENV = colab_setup.drive_env(DRIVE_DIR)
os.environ.update(ENV)

In [ ]:
# 4. The sound-effect library: downloaded once into your Drive, skipped when it is already there.
!cd {REPO} && python scripts/build_sfx_library.py

In [ ]:
# 5. Start the app and open the link it prints (the first load can take a few seconds).
app, tunnel, url = colab_setup.launch(REPO, ENV)
print('Open VibeCrop AI:', url)

## If something goes wrong

- **No link / the page does not load:** run cell 5 again (it starts a fresh link).
- **"Missing GROQ_API_KEY" in the app:** the secret is not set or Notebook access is off (cell 3 tells you).
- **Run out of space:** the app deletes its temporary files after each render; *Start a new video* also cleans up.
- **To stop:** Runtime > Disconnect and delete runtime. Your Drive folder (sounds, model) stays.